In [ ]:
# In bash: pip install scvi-tools

# 1. Set up data:

In [ ]:
import scvi
import scanpy as sc

# Point scVI to raw counts — critical
# If raw counts are in a layer:
scvi.model.SCVI.setup_anndata(
    adata,
    layer="counts",          # raw counts layer
    batch_key="patient_id"   # tells scVI to correct for patient
)

# If .X is already your best option:
scvi.model.SCVI.setup_anndata(
    adata,
    batch_key="patient_id"
)

# 2. Build and Train the model:

In [ ]:
model = scvi.model.SCVI(
    adata,
    n_latent=20,        # dimensions in latent space, start with 20
    n_layers=2,         # depth of encoder/decoder networks
    n_hidden=128        # width of each layer
)

model.train(
    max_epochs=400,
    early_stopping=True  # stops when validation loss plateaus
)

# 3. Extact the latent representations: 

In [ ]:
# Get the 20-dimensional coordinates for every cell
latent = model.get_latent_representation()  # shape: (n_cells, 20)

# Store back in AnnData for downstream use
adata.obsm["X_scVI"] = latent

# 5. Build new UMAP from scVI coordinates

In [ ]:
# Recompute neighbors using scVI space instead of PCA
sc.pp.neighbors(adata, use_rep="X_scVI")
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

sc.pl.umap(adata, color=["leiden", "patient_id", "cell_type"])

# Interpreting the Gene Programs
This is where it gets biologically interesting. Each of the 20 latent dimensions captures a gene program. To find out what each one means:

In [ ]:
# Get the decoder weights — these link latent dims to genes
# scVI has a built-in method for this
df = model.differential_expression(
    adata,
    groupby="leiden"     # compare clusters in latent space
)

A more direct approach to gene programs — using the latent dimensions themselves:

In [ ]:
import pandas as pd
import numpy as np

# Correlate each latent dimension with gene expression
# to find which genes "belong" to each program
latent_df = pd.DataFrame(latent, columns=[f"program_{i}" for i in range(20)])

# For each program, find top genes
for prog in latent_df.columns:
    correlations = np.corrcoef(latent_df[prog], adata.X.toarray().T)[0, 1:]
    top_genes = adata.var_names[np.argsort(correlations)[-20:]]
    print(f"{prog}: {list(top_genes)}")

# The Training Loss — How You Know It's Working

In [ ]:
# Plot training history
model.history["elbo_train"]   # should decrease and plateau
model.history["elbo_validation"]  # should track training loss closely

If validation loss diverges from training loss, you're overfitting — reduce n_latent or increase early_stopping stringency. With your cell numbers this is unlikely to be a problem.